In [6]:
# Imports
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.svm import SVC

In [7]:
# Retrieve metric names
from sklearn.metrics import get_scorer_names, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report

get_scorer_names()

['accuracy',
 'adjusted_mutual_info_score',
 'adjusted_rand_score',
 'average_precision',
 'balanced_accuracy',
 'completeness_score',
 'd2_absolute_error_score',
 'd2_brier_score',
 'd2_log_loss_score',
 'explained_variance',
 'f1',
 'f1_macro',
 'f1_micro',
 'f1_samples',
 'f1_weighted',
 'fowlkes_mallows_score',
 'homogeneity_score',
 'jaccard',
 'jaccard_macro',
 'jaccard_micro',
 'jaccard_samples',
 'jaccard_weighted',
 'matthews_corrcoef',
 'mutual_info_score',
 'neg_brier_score',
 'neg_log_loss',
 'neg_max_error',
 'neg_mean_absolute_error',
 'neg_mean_absolute_percentage_error',
 'neg_mean_gamma_deviance',
 'neg_mean_poisson_deviance',
 'neg_mean_squared_error',
 'neg_mean_squared_log_error',
 'neg_median_absolute_error',
 'neg_negative_likelihood_ratio',
 'neg_root_mean_squared_error',
 'neg_root_mean_squared_log_error',
 'normalized_mutual_info_score',
 'positive_likelihood_ratio',
 'precision',
 'precision_macro',
 'precision_micro',
 'precision_samples',
 'precision_weighte

In [8]:
# Read dataframe
df = pd.read_csv("../data/raw/train.csv", index_col = 'id')
df

,Age,Annual_Income_USD,Daily_Commute_km,Number_of_Cars_Owned,Charging_Stations_Near_Home,Charging_Stations_Near_Work,Environmental_Concern_Level,Gender,City_Type,Current_Car_Type,Home_Charging_Possible,Subsidy_Available,Range_Anxiety_Level,Will_Buy_EV
id,,,,,,,,,,,,,,
0,66,92887.0,23.4,2,3,7,1.0,Male,Suburban,Sedan,Yes,No,Low,No
1,38,30000.0,5.0,1,2,2,4.0,Male,Rural,SUV,Yes,No,Low,No
2,26,94389.0,36.8,1,8,15,5.0,Female,Urban,Sedan,No,Yes,Low,Yes
3,66,73580.0,23.7,2,6,9,3.0,Male,Suburban,Hatchback,Yes,No,Low,No
4,54,57898.0,50.8,1,2,3,3.0,Male,Suburban,Hatchback,Yes,No,Low,No
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
668660,30,71090.0,27.4,2,5,6,5.0,Male,Urban,Sedan,No,Yes,Medium,No
668661,32,129990.0,5.0,2,5,4,1.0,Female,Suburban,SUV,Yes,Yes,Low,No
668662,64,121791.0,24.2,1,5,6,1.0,Female,Suburban,Hatchback,Yes,Yes,Low,No


In [ ]:
# Feature Engineering

def feature_engineering(df):
    df["Total_Nearby_Charging"] = (
        df["Charging_Stations_Near_Home"]
        + df["Charging_Stations_Near_Work"]
    )
    df["Income_per_Car"] = (
        df["Annual_Income_USD"] /
        (df["Number_of_Cars_Owned"] + 1)
    )

    df["Charging_per_Commute"] = (
        df["Total_Nearby_Charging"] /
        (df["Daily_Commute_km"] + 1)
    )
    
    return df

In [30]:
# Select data

X = df.drop(columns = ['Will_Buy_EV'])
y = df['Will_Buy_EV'].map({'Yes': 1,
                           'No': 0})


numerical_columns = X.select_dtypes(include = 'number').columns
nominal_columns = X.select_dtypes(exclude = 'number').drop(columns = ['Range_Anxiety_Level']).columns
ordinal_columns = ['Range_Anxiety_Level']

In [31]:
# Train Test Split

X_train, X_test, y_train, y_test = train_test_split(X,
                                                    y,
                                                    test_size = 0.2,
                                                    shuffle = True,
                                                    stratify = y,
                                                    random_state = 42)

In [32]:
# Preprocessing pipelines

numerical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy = 'mean')),
    ('scaler', StandardScaler())
])

nominal_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy = 'most_frequent')),
    ('scaler', OneHotEncoder())
])

ordinal_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy = 'most_frequent')),
    ('scaler', OrdinalEncoder())
])

preprocessing = ColumnTransformer([
    ('numerical', numerical_pipeline, numerical_columns),
    ('nominal', nominal_pipeline, nominal_columns),
    ('ordinal', ordinal_pipeline, ordinal_columns)
])

In [33]:
# Model Instantiation

models = {
    'Linear': LogisticRegression(),
    'SGD': SGDClassifier(loss = 'log_loss'),
    'HistGradientBoosting': HistGradientBoostingClassifier()
}

In [34]:
# Cross-Validation

cv = StratifiedKFold(
    n_splits = 5,
    shuffle = True,
    random_state = 42
)

model_score = {}

for name, model in models.items():
    pipeline = Pipeline([
        ('preprocessor', preprocessing),
        ('model', model)
    ])

    search = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv = cv,
        n_jobs = -1,
        verbose = True,
        scoring = {'accuracy': 'accuracy',
                   'precision': 'precision_weighted',
                   'recall': 'recall_weighted',
                   'f1': 'f1_weighted',
                   'roc_auc': 'roc_auc_ovr'} # One-vs-Rest ROC-AUC strategy
    )

    model_score[name] = {
        'accuracy': search['test_accuracy'].mean(),
        'precision': search['test_precision'].mean(),
        'recall': search['test_recall'].mean(),
        'f1': search['test_f1'].mean(),
        'ROC-AUC': search['test_roc_auc'].mean()
    }



[Parallel(n_jobs=-1)]: Using backend LokyBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Done   5 out of   5 | elapsed:    8.7s finished
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Done   5 out of   5 | elapsed:    5.4s finished
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Done   5 out of   5 | elapsed:   21.1s finished


In [35]:
# Cross-Validation Results
df_cv = pd.DataFrame(model_score).T
df_cv

,accuracy,precision,recall,f1,ROC-AUC
Linear,0.894443,0.891271,0.894443,0.892553,0.937541
SGD,0.893970,0.890928,0.893970,0.892092,0.937195
HistGradientBoosting,0.897897,0.894956,0.897897,0.896142,0.941003


In [36]:
# Select promising models from CV
promising_models = {
    'HistGradientBoosting': models['HistGradientBoosting'],
    'Linear': models['Linear']
}

In [37]:
# Configure the parameters for each model

rng = np.random.default_rng(seed = 1)

param_distributions = {
    "HistGradientBoosting": {
        "model__learning_rate": rng.uniform(0.01, 0.19, size = 20),
        "model__max_iter": rng.integers(100, 500, size = 20),
        "model__max_leaf_nodes": rng.integers(10, 64, size = 20),
        "model__max_depth": [None, 3, 5, 7, 10],
        "model__min_samples_leaf": rng.integers(10, 50, size = 20),
        "model__l2_regularization": np.exp(
            rng.uniform(
                np.log(1e-3),
                np.log(10),
                size=20)
            )
    },
    "Linear": {
        "model__C": np.exp(
            rng.uniform(
                np.log(1e-3),
                np.log(100),
                size=20,)
            ),
        "model__solver": ["lbfgs", "liblinear"],
        "model__max_iter": [500, 1000, 2000]
    }
}

# So roughly 10% of samples will be below 1, while roughly 90% will be above 1.

# That's why I said it would "overwhelmingly" generate larger values.

# Why log-uniform is different

# For regularisation, we might care about values like:

# 0.001
# 0.01
# 0.1
# 1
# 10

# Notice that these aren't evenly spaced numerically. They're evenly spaced in terms of multiplication by 10.

# A log-uniform distribution treats these ranges roughly equally:

# 0.001 ── 0.01 ── 0.1 ── 1 ── 10
#    ×10      ×10      ×10    ×10

# So you get approximately equal chances of landing in:

# 0.001–0.01
# 0.01–0.1
# 0.1–1
# 1–10

# Uniform:

# "Give every numerical distance the same probability."

# 0    1    2    3    4    5
# |----|----|----|----|----|

# Log-uniform:

# "Give every multiplicative scale the same probability."

# 0.001   0.01   0.1    1     10
#    |------|------|------|------|

In [38]:
# Hyperparameter optimisation

tuned_models = {}

for name, model in promising_models.items():

    pipeline = Pipeline([
        ('preprocessor', preprocessing),
        ('model', model)
    ])

    search = RandomizedSearchCV(
        pipeline,
        param_distributions = param_distributions[name],
        n_iter = 10,
        n_jobs = 4,
        scoring = 'roc_auc',
        verbose = True,
        random_state = 42
    )

    search.fit(X_train, y_train)

    tuned_model = search.best_estimator_
    tuned_models[name] = {
        'model': tuned_model,
        'roc_auc': search.best_score_
    }


Fitting 5 folds for each of 10 candidates, totalling 50 fits
Fitting 5 folds for each of 10 candidates, totalling 50 fits


In [49]:
# for name in tuned_models.items():
#     print(name)
# Output model scores after hyperparameter optimisation
df_hyperparameter_optimised = pd.DataFrame(tuned_models).T
df_hyperparameter_optimised

,model,roc_auc
HistGradientBoosting,"(ColumnTransformer(transformers=[('numerical',...",0.941543
Linear,"(ColumnTransformer(transformers=[('numerical',...",0.937536


In [ ]:
# Select final model
final_model = tuned_models['HistGradientBoosting']['model']
predictions = final_model.predict(X_test)

In [69]:
# Scoring model
print(confusion_matrix(y_test, predictions))

classification_report_df = pd.DataFrame(classification_report(y_test, predictions, output_dict = True))
classification_report_df

[[104336   6041]
 [  7652  15704]]


,0,1,accuracy,macro avg,weighted avg
precision,0.931671,0.722189,0.897609,0.826930,0.895086
recall,0.945269,0.672375,0.897609,0.808822,0.897609
f1-score,0.938421,0.696393,0.897609,0.817407,0.896152
support,110377.000000,23356.000000,0.897609,133733.000000,133733.000000


In [70]:
# Scoring model
scoring_df = pd.DataFrame({
    'Accuracy': [accuracy_score(y_test, predictions)],
    'Precision': [precision_score(y_test, predictions)],
    'Recall': [recall_score(y_test, predictions)],
    'f1': [f1_score(y_test, predictions)],
    'ROC-AUC': [roc_auc_score(y_test, predictions)]
}, index = ['HistGradientBoosting'])

scoring_df

,Accuracy,Precision,Recall,f1,ROC-AUC
HistGradientBoosting,0.897609,0.722189,0.672375,0.696393,0.808822


In [42]:
import joblib

joblib.dump(final_model, "../models/model.pkl")

['../models/model.pkl']

In [53]:
# Select data

# Read dataframe

X = pd.read_csv("../data/raw/test.csv", index_col = 'id')
X = feature_engineering(X)

model = joblib.load("../models/model.pkl")
predictions = model.predict(X)

df_result = pd.DataFrame({
    'Will_Buy_EV': predictions
}, index = X.index)
df_result.to_csv("../submissions/submission.csv")
